# Llama 3.2 3B — Local Inference with Ollama (OpenAI-compatible)

Runs entirely on your machine — no API key, no internet after the first pull.

**Prerequisites:** [Ollama](https://ollama.com) must be installed and running (`ollama serve`).

## Setup

In [ ]:
%pip install openai -q

In [5]:
# Pull the model once via terminal before running the notebook:
#   ollama pull llama3.2:3b

from openai import OpenAI

MODEL = "qwen2.5:3b"  # 2.0 GB — fast, fits on most laptops

client = OpenAI(
    api_key  = "ollama",                  # any non-empty string
    base_url = "http://localhost:11434/v1"
)

---
## 1 — Q&A

In [7]:
question = "What is the difference between supervised and unsupervised learning?"

response = client.chat.completions.create(
    model       = MODEL,
    messages    = [
        {"role": "user", "content": question}
    ],
    temperature = 0.0,
    max_tokens  = 128
)

print(response.choices[0].message.content)

Supervised and unsupervised learning are two fundamental paradigms in machine learning that differ in their approach to training models and how they utilize data.

### Supervised Learning

In supervised learning, the model is trained using a dataset that includes both input features and corresponding output labels. The goal of the model is to learn the mapping between these inputs and outputs so it can make predictions on new, unseen data with similar characteristics.

Key Characteristics:
- **Labeled Data**: The training set contains examples where each example has an associated label or target value.
- **Training Process**: The algorithm learns from a dataset that includes both input features (X


In [8]:
question2 = "Explain what a transformer architecture is in simple terms."

response2 = client.chat.completions.create(
    model       = MODEL,
    messages    = [
        {"role": "user", "content": question2}
    ],
    temperature = 0.7,
    max_tokens  = 512
)

print(response2.choices[0].message.content)

A transformer architecture is a type of neural network used for natural language processing (NLP) tasks and other sequence-to-sequence learning problems. Unlike traditional recurrent neural networks (RNNs), which are good at handling sequential data but suffer from the vanishing gradient problem, transformers use self-attention mechanisms to process information in parallel.

Here's a simple way to think about it:

1. **Parallel Processing**: Instead of sequentially processing each word or token like RNNs do, transformers can look at every pair of words (or tokens) simultaneously. This allows them to understand the context between any two points in the sequence very quickly and efficiently.

2. **Self-Attention Mechanism**: One key component is the self-attention mechanism. Imagine you have a sentence: "The quick brown fox jumps over the lazy dog." If we're trying to understand an important word like "quick," transformers can prioritize it by understanding its relationship with other wo

---
## 2 — System Prompt

In [9]:
system_prompt = """
You are a Python programming tutor for university students.
You only answer questions related to Python.
If the user asks about anything else, politely decline and redirect them to Python.
Keep answers concise and always include a short code example.
"""

user_question = "How do list comprehensions work?"

response = client.chat.completions.create(
    model       = MODEL,
    messages    = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_question}
    ],
    temperature = 0.7,
    max_tokens  = 512
)

print(response.choices[0].message.content)

List comprehensions in Python provide a succinct way to create lists based on existing iterables. They consist of brackets containing an expression followed by a `[ ]` clause; for those familiar with other programming languages, it can be seen as being similar to array formula syntax.

Here is a simple example:
```python
squares = [x**2 for x in range(5)]
print(squares)  # Output: [0, 1, 4, 9, 16]
```

This list comprehension generates a list of squares of numbers from 0 to 4.


In [10]:
off_topic = "Can you explain the French Revolution?"

response_off = client.chat.completions.create(
    model       = MODEL,
    messages    = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": off_topic}
    ],
    temperature = 0.7,
    max_tokens  = 512
)

print(response_off.choices[0].message.content)

Certainly, I can provide information on the French Revolution. However, I'll focus on Python as per your request. Let's move forward with that. How would you like to proceed?


---
## 3 — Multi-Turn Chat

In [11]:
def chat(history, user_message):
    history.append({"role": "user", "content": user_message})
    response = client.chat.completions.create(
        model       = MODEL,
        messages    = history,
        temperature = 0.7,
        max_tokens  = 512
    )
    assistant_reply = response.choices[0].message.content
    history.append({"role": "assistant", "content": assistant_reply})
    return assistant_reply, history

In [12]:
history = []

reply, history = chat(history, "Hi! I want to learn about neural networks. Where should I start?")
print("Turn 1:\n", reply, "\n")

reply, history = chat(history, "What is backpropagation exactly?")
print("Turn 2:\n", reply, "\n")

reply, history = chat(history, "Can you give me a very simple analogy for it?")
print("Turn 3:\n", reply)

Turn 1:
 Great! Learning about neural networks can be a fun and rewarding journey. Here’s a step-by-step guide to help you get started:

### 1. Understand the Basics of Machine Learning
Before diving into deep learning, it's important to understand the basics of machine learning. This includes concepts like:
- **Supervised vs Unsupervised Learning**
- **Classification vs Regression**
- **Linear Regression**
- **Decision Trees and Random Forests**

These foundational concepts will give you a strong understanding of what neural networks are trying to achieve.

### 2. Learn About Neural Networks
Once you have the basics covered, you can start learning about neural networks:
- **Neural Network Basics**: Understand how neurons in a network work together.
- **Types of Neural Networks**: Feedforward Neural Networks, Convolutional Neural Networks (CNNs), Recurrent Neural Networks (RNNs), etc.
- **Activation Functions**: Sigmoid, Tanh, ReLU, and more.

### 3. Dive into Mathematics
Deep learning

In [13]:
for msg in history:
    print(f"[{msg['role'].upper()}]")
    print(msg['content'][:200], "..." if len(msg['content']) > 200 else "")
    print()

[USER]
Hi! I want to learn about neural networks. Where should I start? 

[ASSISTANT]
Great! Learning about neural networks can be a fun and rewarding journey. Here’s a step-by-step guide to help you get started:

### 1. Understand the Basics of Machine Learning
Before diving into deep ...

[USER]
What is backpropagation exactly? 

[ASSISTANT]
Backpropagation is an essential algorithm used in training artificial neural networks. Essentially, it's a method for efficiently computing the gradient of the loss function with respect to the weight ...

[USER]
Can you give me a very simple analogy for it? 

[ASSISTANT]
Certainly! Let's use an analogy involving baking cookies to understand backpropagation.

### The Cookie-Baking Analogy

Imagine you're baking chocolate chip cookies. Your goal is to maximize the delic ...



---
## 4 — Summarization

In [ ]:
long_text = """
Large language models (LLMs) are a type of artificial intelligence model trained on vast amounts of
text data. They use a transformer architecture with billions of parameters, allowing them to understand
and generate human-like text. Training involves predicting the next token in a sequence, which forces
the model to learn grammar, facts, reasoning, and even some degree of common sense from the data.

After pretraining, models are typically fine-tuned using techniques such as Reinforcement Learning
from Human Feedback (RLHF), which aligns the model's outputs with human preferences and makes it
more helpful and less harmful. Popular LLMs include GPT-4, Claude, Gemini, and LLaMA.

Despite their capabilities, LLMs have notable limitations: they can hallucinate facts, lack true
understanding, have a training knowledge cutoff, and can reflect biases present in their training data.
Research into improving reliability, reducing hallucinations, and enabling longer context windows
is ongoing.
"""

response = client.chat.completions.create(
    model       = MODEL,
    messages    = [
        {"role": "system", "content": "Summarize the given text in 3 bullet points. Be concise."},
        {"role": "user",   "content": long_text}
    ],
    temperature = 0.3,
    max_tokens  = 256
)

print(response.choices[0].message.content)

In [ ]:
response_simple = client.chat.completions.create(
    model       = MODEL,
    messages    = [
        {"role": "system", "content": "Summarize the given text as if explaining to a 10-year-old. Use simple words."},
        {"role": "user",   "content": long_text}
    ],
    temperature = 0.7,
    max_tokens  = 256
)

print(response_simple.choices[0].message.content)

---
## 5 — Streaming (bonus)

See tokens appear as they are generated instead of waiting for the full reply.

In [14]:
stream = client.chat.completions.create(
    model       = MODEL,
    messages    = [{"role": "user", "content": "Tell me a fun fact about machine learning."}],
    temperature = 0.7,
    max_tokens  = 256,
    stream      = True
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

Sure! Here's a fun fact for you:

The first neural network to achieve human-level performance in the ImageNet Large Scale Visual Recognition Challenge (ILSVRC) was developed in 2012 by Alex Krizhevker and Ilya Sutskevitch, but it wasn't until 2012 that AlexNet achieved a 76.5% accuracy on the task, which is considered human-level performance at the time.

What makes this fun fact interesting is how deep learning models have evolved significantly since then. In the years following AlexNet's success, we've seen tremendous advancements in the field of machine learning and computer vision, with many state-of-the-art models achieving remarkable results on a wide range of tasks, from image classification to natural language processing.

Isn't it fascinating how technology has advanced so much?

---
## 6 — Check finish_reason

If `finish_reason` is `"length"` the response was cut off by `max_tokens`. If it is `"stop"` the model finished naturally.

In [15]:
response = client.chat.completions.create(
    model       = MODEL,
    messages    = [{"role": "user", "content": "Explain neural networks in detail."}],
    temperature = 0.7,
    max_tokens  = 50   # intentionally small to trigger truncation
)

print("finish_reason:", response.choices[0].finish_reason)
print()
print(response.choices[0].message.content)

finish_reason: length

Neural networks are computational models inspired by the structure and function of biological neurons in the human brain. They are used for various applications such as pattern recognition, predictive modeling, decision making, and much more.

### Basic Structure

A Neural Network is typically
